# Call GPT — Gauge Video Inference

Runs GPT on all videos under `Dataset_0508/<GaugeName>/` (all subdirectories).  
OpenAI does not support direct video upload — frames are extracted via OpenCV at 200ms intervals and passed as an image array.  
**Only edit the CONFIG cell.**

In [5]:
# ============================
# CONFIG — edit this cell only
# ============================
GAUGE_IDS        = [1, 2, 3]                                              # gauges to run
MODEL_NAMES      = ["gpt-5.4-2026-03-05", "gpt-5.3-chat-latest"]         # models to run
STRATEGIES       = ["cot", "naive"]                                       # prompt strategies
REASONING_EFFORT = "high"                                                 # none/low/medium/high/xhigh — only for gpt-5.4

FRAME_INTERVAL_SEC = 1.0                                                  # 1 frame per second, consistent with Gemini fps=1

In [6]:
import os, time, base64
from itertools import product
import cv2
import pandas as pd
from openai import OpenAI

_GAUGE_CONFIG = {
    1: {
        "name":      "1.- Analog Dial Gauge",
        "video_dir": "Dataset_0508/1.- Analog Dial Gauge",
        "label_ref": "Dataset_0508/metadata/gauge_1_metadata.xlsx",
    },
    2: {
        "name":      "2.- Analog Depth Gauge",
        "video_dir": "Dataset_0508/2.- Analog Depth Gauge",
        "label_ref": "Dataset_0508/metadata/gauge_2_metadata.xlsx",
    },
    3: {
        "name":      "3.- Analog Depth Gauge_without vernier",
        "video_dir": "Dataset_0508/3.- Analog Depth Gauge_without vernier",
        "label_ref": "Dataset_0508/metadata/gauge_3_metadata.xlsx",
    },
}

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def extract_frames_as_base64(video_path, interval_sec=0.2, resize_width=540):
    cap = cv2.VideoCapture(video_path)
    frames_out = []
    t_sec = 0.0
    while True:
        cap.set(cv2.CAP_PROP_POS_MSEC, t_sec * 1000)
        ret, frame = cap.read()
        if not ret:
            break
        h, w = frame.shape[:2]
        resize_height = int(h * resize_width / w)
        frame = cv2.resize(frame, (resize_width, resize_height))
        _, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
        b64 = base64.b64encode(buf.tobytes()).decode("utf-8")
        frames_out.append((int(t_sec * 1000), b64))
        t_sec += interval_sec
    cap.release()
    return frames_out


MAX_RETRIES = 5
RETRY_DELAY = 10  # seconds

for GAUGE_ID, MODEL_NAME, PROMPT_STRATEGY in product(GAUGE_IDS, MODEL_NAMES, STRATEGIES):
    print(f"\n{'#'*60}")
    print(f"# GAUGE={GAUGE_ID}  MODEL={MODEL_NAME}  STRATEGY={PROMPT_STRATEGY}")
    print(f"{'#'*60}")

    cfg          = _GAUGE_CONFIG[GAUGE_ID]
    naive_suffix = "_naive" if PROMPT_STRATEGY == "naive" else ""
    output_dir   = f"Results_0508/GPT/{cfg['name']}_{MODEL_NAME}{naive_suffix}"
    os.makedirs(output_dir, exist_ok=True)

    _meta = pd.read_excel(cfg["label_ref"], sheet_name="Metadata")
    _meta.set_index("Column (Header)", inplace=True)
    gauge_type          = _meta.loc["type",                    "Content / Example"]
    unit                = _meta.loc["unit",                    "Content / Example"]
    reading_start       = float(_meta.loc["manufacturer_range_min", "Content / Example"])
    reading_end         = float(_meta.loc["manufacturer_range_max", "Content / Example"])
    graduation_interval = _meta.loc["graduation_interval",    "Content / Example"]

    # gpt-5.4 supports reasoning effort; gpt-5.3-chat-latest does not (and rejects temperature)
    _is_thinking_model = "5.4" in MODEL_NAME

    video_files = []
    for dirpath, _, filenames in os.walk(cfg["video_dir"]):
        for fname in sorted(filenames):
            if fname.endswith(".mp4"):
                video_files.append(os.path.join(dirpath, fname))
    print(f"Found {len(video_files)} videos under {cfg['video_dir']}")

    # ============================
    # PROMPTS
    # ============================
    _cot_message = f'''ROLE: Expert Industrial Metrology Assistant

PROTOCOL: See-Think-Confirm
1. SEE: Localize gauge boundaries. Identify [0%] and [100%] markers.
2. THINK: Synchronize needle position with the in-band digital clock. Calculate velocity (ΔReading/ΔTime). Use 'Thought Signatures' to maintain temporal state between frames.
3. CONFIRM: Verify that the reading is within [min, max] and follows physical monotonicity relative to previous frames.
CONSTRAINT: Output JSON only. Format: {{"ts_ms": integer, "reading": float, "confidence": float}}
TASK: Extract a high-precision time-series of readings from the video.

INSTRUMENT METADATA:
- Gauge Type: {gauge_type} | Unit: {unit}
- Graduation Interval: {graduation_interval}
- Calibrated Range: [{reading_start}, {reading_end}]

SAMPLING PROTOCOL:
- Absolute Time Reference: Every 'ts_ms' in your output must correspond to the EXACT numerical value shown on the digital chronometer in the frame.
- Starting Point: The chronometer in this video starts at a non-zero value. Ignore any frames before the timer reaches the first 200ms integer increment (e.g., if the timer starts at 5670ms, your first entry should be at 5800ms).
- Sampling Interval: Provide one reading every 200ms based on the chronometer's increments (e.g., 5800, 6000, 6200...).
- Duration: Continue this sequence until the video ends, strictly following the chronometer's values, regardless of the video file's elapsed time.

RESPONSE REQUIREMENT:
Output a single JSON array where each entry contains: {{"ts_ms": integer, "reading": float, "confidence": float}}'''

    _naive_message = f'''ROLE: Expert Industrial Metrology Assistant

TASK: Read the analog gauge values from the video at specific time intervals.

INSTRUMENT METADATA:
- Gauge Type: {gauge_type} | Unit: {unit}
- Graduation Interval: {graduation_interval}
- Calibrated Range: [{reading_start}, {reading_end}]

SAMPLING REQUIREMENTS:
- Use the digital chronometer shown in the video for timing.
- Start reading from the first 200ms integer increment.
- Provide one reading every 200ms (e.g., 5800, 6000, 6200...).
- Continue until the video ends.

OUTPUT FORMAT:
Return only a JSON array of objects: {{"ts_ms": integer, "reading": float, "confidence": float}}'''

    message = _cot_message if PROMPT_STRATEGY == "cot" else _naive_message
    print(message)

    # ============================
    # INFERENCE LOOP
    # ============================
    for video_path in video_files:
        rel    = os.path.relpath(video_path, cfg["video_dir"])
        seq_id = rel.replace(os.sep, "_").replace(".mp4", "")

        output_file = os.path.join(output_dir, f"{seq_id}_Raw_Results.xlsx")
        if os.path.exists(output_file):
            print(f"Skipping: {seq_id}")
            continue

        print(f"\n{'='*20} Processing: {seq_id} {'='*20}")
        program_start_time = time.time()

        print("Extracting frames...")
        frames = extract_frames_as_base64(video_path, interval_sec=FRAME_INTERVAL_SEC)
        print(f"Extracted {len(frames)} frames.")

        content = [{"type": "input_text", "text": message}]
        for _, b64 in frames:
            content.append({"type": "input_image", "image_url": f"data:image/jpeg;base64,{b64}"})

        kwargs = dict(
            model=MODEL_NAME,
            input=[{"role": "user", "content": content}],
        )
        if _is_thinking_model:
            kwargs["reasoning"] = {"effort": REASONING_EFFORT}

        print("Generating...")
        response = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = client.responses.create(**kwargs)
                break
            except Exception as e:
                print(f"Attempt {attempt}/{MAX_RETRIES} failed: {e}")
                if attempt < MAX_RETRIES:
                    print(f"Retrying in {RETRY_DELAY}s...")
                    time.sleep(RETRY_DELAY)
                else:
                    print(f"All retries exhausted. Skipping {seq_id}.")

        if response is None:
            continue

        response_text = ""
        if hasattr(response, "output_text"):
            response_text = response.output_text or ""
        if not response_text and hasattr(response, "output"):
            for item in response.output:
                if getattr(item, "type", None) == "message":
                    for part in getattr(item, "content", []):
                        if getattr(part, "type", None) == "output_text":
                            response_text = getattr(part, "text", "") or response_text
        print(response_text)

        total_duration = time.time() - program_start_time
        pd.DataFrame([{
            "sequence_id":        seq_id,
            "raw_model_response": response_text,
            "total_duration_sec": round(total_duration, 2),
            "model_name":         MODEL_NAME,
        }]).to_excel(output_file, index=False)
        print(f"Saved: {output_file}")


############################################################
# GAUGE=1  MODEL=gpt-5.4-2026-03-05  STRATEGY=cot
############################################################
Found 18 videos under Dataset_0508/1.- Analog Dial Gauge
ROLE: Expert Industrial Metrology Assistant

PROTOCOL: See-Think-Confirm
1. SEE: Localize gauge boundaries. Identify [0%] and [100%] markers.
2. THINK: Synchronize needle position with the in-band digital clock. Calculate velocity (ΔReading/ΔTime). Use 'Thought Signatures' to maintain temporal state between frames.
3. CONFIRM: Verify that the reading is within [min, max] and follows physical monotonicity relative to previous frames.
CONSTRAINT: Output JSON only. Format: {"ts_ms": integer, "reading": float, "confidence": float}
TASK: Extract a high-precision time-series of readings from the video.

INSTRUMENT METADATA:
- Gauge Type: Circular | Unit: mm
- Graduation Interval: 0.01
- Calibrated Range: [0.0, 10.0]

SAMPLING PROTOCOL:
- Absolute Time Reference: Eve

Extracted 35 frames.
Generating...
[{"ts_ms":2000,"reading":0.0,"confidence":0.72},{"ts_ms":2800,"reading":0.0,"confidence":0.72},{"ts_ms":3800,"reading":0.0,"confidence":0.72},{"ts_ms":4800,"reading":0.02,"confidence":0.7},{"ts_ms":5800,"reading":0.9,"confidence":0.75},{"ts_ms":6800,"reading":0.8,"confidence":0.75},{"ts_ms":7800,"reading":0.7,"confidence":0.75},{"ts_ms":8800,"reading":0.55,"confidence":0.74},{"ts_ms":9800,"reading":0.5,"confidence":0.74},{"ts_ms":10800,"reading":0.35,"confidence":0.73},{"ts_ms":11800,"reading":0.2,"confidence":0.73},{"ts_ms":12800,"reading":0.15,"confidence":0.72},{"ts_ms":13800,"reading":0.1,"confidence":0.72},{"ts_ms":14800,"reading":0.0,"confidence":0.72},{"ts_ms":15800,"reading":0.9,"confidence":0.75},{"ts_ms":16800,"reading":0.8,"confidence":0.75},{"ts_ms":17800,"reading":0.7,"confidence":0.75},{"ts_ms":18800,"reading":0.55,"confidence":0.74},{"ts_ms":19800,"reading":0.5,"confidence":0.74},{"ts_ms":20800,"reading":0.35,"confidence":0.73},{"ts_ms"

Extracted 11 frames.
Generating...
[{"ts_ms":1770,"reading":0.0,"confidence":0.62},{"ts_ms":2760,"reading":0.0,"confidence":0.65},{"ts_ms":3770,"reading":0.0,"confidence":0.66},{"ts_ms":4760,"reading":0.2,"confidence":0.7},{"ts_ms":5770,"reading":0.3,"confidence":0.72},{"ts_ms":6760,"reading":0.32,"confidence":0.68},{"ts_ms":7760,"reading":0.4,"confidence":0.73},{"ts_ms":8760,"reading":0.45,"confidence":0.72},{"ts_ms":9760,"reading":0.5,"confidence":0.74},{"ts_ms":10750,"reading":1.0,"confidence":0.78},{"ts_ms":11760,"reading":1.02,"confidence":0.69}]
Saved: Results_0508/GPT/1.- Analog Dial Gauge_gpt-5.3-chat-latest\DOWN_DOWN_V5_50 divisions_per_second_Raw_Results.xlsx

==================== Processing: DOWN_DOWN_V6_100 divisions_per_second ====================
Extracting frames...
Extracted 9 frames.
Generating...
[{"ts_ms":2000,"reading":0.0,"confidence":0.92},{"ts_ms":3000,"reading":0.0,"confidence":0.9},{"ts_ms":4000,"reading":0.0,"confidence":0.9},{"ts_ms":5000,"reading":0.15,"conf

Extracted 11 frames.
Generating...
[{"ts_ms":1450,"reading":0.0,"confidence":0.92},{"ts_ms":2460,"reading":0.0,"confidence":0.92},{"ts_ms":3450,"reading":0.0,"confidence":0.9},{"ts_ms":4460,"reading":0.6,"confidence":0.75},{"ts_ms":5460,"reading":0.85,"confidence":0.75},{"ts_ms":6450,"reading":0.7,"confidence":0.72},{"ts_ms":7460,"reading":0.8,"confidence":0.72},{"ts_ms":8450,"reading":0.7,"confidence":0.7},{"ts_ms":9460,"reading":0.78,"confidence":0.7},{"ts_ms":10450,"reading":0.1,"confidence":0.68},{"ts_ms":11440,"reading":0.05,"confidence":0.68}]}
Saved: Results_0508/GPT/1.- Analog Dial Gauge_gpt-5.3-chat-latest\UP_UP_V5_50 divisions_per_second_Raw_Results.xlsx

==================== Processing: UP_UP_V6_100 divisions_per_second ====================
Extracting frames...
Extracted 8 frames.
Generating...
[{"ts_ms":1510,"reading":0.0,"confidence":0.9},{"ts_ms":2500,"reading":0.0,"confidence":0.9},{"ts_ms":3510,"reading":0.0,"confidence":0.9},{"ts_ms":4500,"reading":0.55,"confidence":0.

Extracted 66 frames.
Generating...
[{"ts_ms":1600,"reading":0.00,"confidence":0.62},{"ts_ms":2600,"reading":0.02,"confidence":0.60},{"ts_ms":3600,"reading":0.05,"confidence":0.60},{"ts_ms":4600,"reading":0.08,"confidence":0.58},{"ts_ms":5600,"reading":0.12,"confidence":0.60},{"ts_ms":6600,"reading":0.18,"confidence":0.62},{"ts_ms":7600,"reading":0.24,"confidence":0.64},{"ts_ms":8600,"reading":0.30,"confidence":0.64},{"ts_ms":9600,"reading":0.36,"confidence":0.65},{"ts_ms":10600,"reading":0.42,"confidence":0.65},{"ts_ms":11600,"reading":0.48,"confidence":0.65},{"ts_ms":12600,"reading":0.55,"confidence":0.66},{"ts_ms":13600,"reading":0.62,"confidence":0.66},{"ts_ms":14600,"reading":0.70,"confidence":0.66},{"ts_ms":15600,"reading":0.78,"confidence":0.66},{"ts_ms":16600,"reading":0.86,"confidence":0.66},{"ts_ms":17600,"reading":0.94,"confidence":0.66},{"ts_ms":18600,"reading":1.02,"confidence":0.66},{"ts_ms":19600,"reading":1.10,"confidence":0.66},{"ts_ms":20600,"reading":1.18,"confidence"

...

Extracted 41 frames.
Generating...
[{"ts_ms":1800,"reading":52.0,"confidence":0.9},{"ts_ms":2000,"reading":52.0,"confidence":0.9},{"ts_ms":2200,"reading":52.0,"confidence":0.9},{"ts_ms":2400,"reading":52.0,"confidence":0.9},{"ts_ms":2600,"reading":52.0,"confidence":0.9},{"ts_ms":2800,"reading":52.0,"confidence":0.9},{"ts_ms":3000,"reading":52.0,"confidence":0.9},{"ts_ms":3200,"reading":52.0,"confidence":0.9},{"ts_ms":3400,"reading":52.0,"confidence":0.9},{"ts_ms":3600,"reading":52.0,"confidence":0.9},{"ts_ms":3800,"reading":52.0,"confidence":0.9},{"ts_ms":4000,"reading":52.0,"confidence":0.9},{"ts_ms":4200,"reading":52.0,"confidence":0.9},{"ts_ms":4400,"reading":52.0,"confidence":0.9},{"ts_ms":4600,"reading":52.0,"confidence":0.9},{"ts_ms":4800,"reading":52.0,"confidence":0.9},{"ts_ms":5000,"reading":52.0,"confidence":0.9},{"ts_ms":5200,"reading":52.0,"confidence":0.9},{"ts_ms":5400,"reading":52.0,"confidence":0.9},{"ts_ms":5600,"reading":52.0,"confidence":0.9},{"ts_ms":5800,"reading":

Extracted 23 frames.
Generating...
[{"ts_ms":2000,"reading":43.0,"confidence":0.6},{"ts_ms":2200,"reading":43.21,"confidence":0.6},{"ts_ms":2400,"reading":43.42,"confidence":0.6},{"ts_ms":2600,"reading":43.63,"confidence":0.6},{"ts_ms":2800,"reading":43.84,"confidence":0.6},{"ts_ms":3000,"reading":44.05,"confidence":0.6},{"ts_ms":3200,"reading":44.26,"confidence":0.6},{"ts_ms":3400,"reading":44.47,"confidence":0.6},{"ts_ms":3600,"reading":44.68,"confidence":0.6},{"ts_ms":3800,"reading":44.89,"confidence":0.6},{"ts_ms":4000,"reading":45.1,"confidence":0.6},{"ts_ms":4200,"reading":45.31,"confidence":0.6},{"ts_ms":4400,"reading":45.52,"confidence":0.6},{"ts_ms":4600,"reading":45.73,"confidence":0.6},{"ts_ms":4800,"reading":45.94,"confidence":0.6},{"ts_ms":5000,"reading":46.15,"confidence":0.6},{"ts_ms":5200,"reading":46.36,"confidence":0.6},{"ts_ms":5400,"reading":46.57,"confidence":0.6},{"ts_ms":5600,"reading":46.78,"confidence":0.6},{"ts_ms":5800,"reading":46.99,"confidence":0.6},{"ts_m

Extracted 7 frames.
Generating...
[{"ts_ms":2000,"reading":42.4,"confidence":0.6},{"ts_ms":2200,"reading":42.54,"confidence":0.6},{"ts_ms":2400,"reading":42.68,"confidence":0.6},{"ts_ms":2600,"reading":42.82,"confidence":0.6},{"ts_ms":2800,"reading":42.96,"confidence":0.6},{"ts_ms":3000,"reading":43.1,"confidence":0.6},{"ts_ms":3200,"reading":43.22,"confidence":0.6},{"ts_ms":3400,"reading":43.34,"confidence":0.6},{"ts_ms":3600,"reading":43.46,"confidence":0.6},{"ts_ms":3800,"reading":43.58,"confidence":0.6},{"ts_ms":4000,"reading":43.7,"confidence":0.6},{"ts_ms":4200,"reading":43.84,"confidence":0.6},{"ts_ms":4400,"reading":43.98,"confidence":0.6},{"ts_ms":4600,"reading":44.12,"confidence":0.6},{"ts_ms":4800,"reading":44.26,"confidence":0.6},{"ts_ms":5000,"reading":44.4,"confidence":0.6},{"ts_ms":5200,"reading":44.54,"confidence":0.6},{"ts_ms":5400,"reading":44.68,"confidence":0.6},{"ts_ms":5600,"reading":44.82,"confidence":0.6},{"ts_ms":5800,"reading":44.96,"confidence":0.6},{"ts_ms":

Extracted 24 frames.
Generating...
[{"ts_ms":2000,"reading":52.2,"confidence":0.8},{"ts_ms":2200,"reading":52.2,"confidence":0.8},{"ts_ms":2400,"reading":52.2,"confidence":0.8},{"ts_ms":2600,"reading":52.2,"confidence":0.8},{"ts_ms":2800,"reading":52.2,"confidence":0.8},{"ts_ms":3000,"reading":52.2,"confidence":0.8},{"ts_ms":3200,"reading":52.2,"confidence":0.8},{"ts_ms":3400,"reading":52.2,"confidence":0.8},{"ts_ms":3600,"reading":52.2,"confidence":0.8},{"ts_ms":3800,"reading":52.2,"confidence":0.8},{"ts_ms":4000,"reading":52.2,"confidence":0.8},{"ts_ms":4200,"reading":52.2,"confidence":0.8},{"ts_ms":4400,"reading":52.2,"confidence":0.8},{"ts_ms":4600,"reading":52.2,"confidence":0.8},{"ts_ms":4800,"reading":52.2,"confidence":0.8},{"ts_ms":5000,"reading":52.2,"confidence":0.8},{"ts_ms":5200,"reading":52.2,"confidence":0.8},{"ts_ms":5400,"reading":52.2,"confidence":0.8},{"ts_ms":5600,"reading":52.2,"confidence":0.8},{"ts_ms":5800,"reading":52.2,"confidence":0.8},{"ts_ms":6000,"reading":

Extracted 12 frames.
Generating...
[{"ts_ms":1800,"reading":52.2,"confidence":0.6},{"ts_ms":2000,"reading":52.21,"confidence":0.6},{"ts_ms":2200,"reading":52.23,"confidence":0.6},{"ts_ms":2400,"reading":52.24,"confidence":0.6},{"ts_ms":2600,"reading":52.25,"confidence":0.6},{"ts_ms":2800,"reading":52.27,"confidence":0.6},{"ts_ms":3000,"reading":52.28,"confidence":0.6},{"ts_ms":3200,"reading":52.29,"confidence":0.6},{"ts_ms":3400,"reading":52.31,"confidence":0.6},{"ts_ms":3600,"reading":52.32,"confidence":0.6},{"ts_ms":3800,"reading":52.33,"confidence":0.6},{"ts_ms":4000,"reading":52.35,"confidence":0.6},{"ts_ms":4200,"reading":52.36,"confidence":0.6},{"ts_ms":4400,"reading":52.37,"confidence":0.6},{"ts_ms":4600,"reading":52.39,"confidence":0.6},{"ts_ms":4800,"reading":52.4,"confidence":0.6},{"ts_ms":5000,"reading":52.41,"confidence":0.6},{"ts_ms":5200,"reading":52.43,"confidence":0.6},{"ts_ms":5400,"reading":52.44,"confidence":0.6},{"ts_ms":5600,"reading":52.45,"confidence":0.6},{"ts_m